#Init

In [0]:
import sqlite3
import pandas as pd

#Listing all the tables from the database

In [0]:
db_path = '/Volumes/tennislakehouse/bronze/source_db/database.sqlite'
conn = sqlite3.connect(db_path)

tables = pd.read_sql_query(
    'SELECT name FROM sqlite_master WHERE type="table";',
    conn
)

table_names = tables['name'].tolist()
print(table_names)

#Loading sqlite tables to Bronze layer

In [0]:
for table in table_names:
    pdf = pd.read_sql_query(f'SELECT * FROM {table}', conn)
    pdf = pdf.drop(columns=["loser2_rank_points"], errors="ignore")
    pdf = pdf.convert_dtypes()

    spark_dataframe = spark.createDataFrame(pdf)

    spark_dataframe.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f'tennislakehouse.bronze.{table}')